# Colab PPO Training with Remote Chip Flooring Env

This notebook runs PPO in Colab while calling your deployed HF Space environment over HTTP.

In [1]:
!pip install -q "transformers>=4.45.0" "trl==0.8.6" "accelerate>=0.33.0" "bitsandbytes" "matplotlib" "requests"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.2 MB/s eta 0:00:00


In [2]:
import ast
import json
import random
import re
import time
from types import SimpleNamespace

import matplotlib.pyplot as plt
import requests
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

HF_SPACE_BASE_URL = "https://gopinathv19-chip-flooring-env.hf.space"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"


In [3]:
class RemoteChipFlooringEnvironment:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        self.block_id_to_index = {}

    def _to_obs(self, payload):
        obs = payload.get("observation", {}) or {}
        obs["reward"] = payload.get("reward", 0.0)
        obs["done"] = payload.get("done", False)
        obs["invalid_reasons"] = payload.get("invalid_reasons")
        return SimpleNamespace(**obs)

    def _refresh_block_index(self, obs):
        blocks = list(getattr(obs, "remaining_blocks", []) or []) + list(getattr(obs, "placed_blocks", []) or [])
        for block in blocks:
            block_id = block.get("id")
            if block_id is not None and block_id not in self.block_id_to_index:
                self.block_id_to_index[block_id] = len(self.block_id_to_index)

    def reset(self, task_name=None):
        payload = {}
        if task_name:
            payload["task_name"] = task_name
        r = self.session.post(f"{self.base_url}/reset", json=payload, timeout=120)
        r.raise_for_status()
        obs = self._to_obs(r.json())
        self.block_id_to_index = {}
        self._refresh_block_index(obs)
        return obs

    def step(self, action):
        r = self.session.post(f"{self.base_url}/step", json={"action": action}, timeout=120)
        r.raise_for_status()
        obs = self._to_obs(r.json())
        self._refresh_block_index(obs)
        return obs

env = RemoteChipFlooringEnvironment(HF_SPACE_BASE_URL)

_ = requests.get(f"{HF_SPACE_BASE_URL}/schema", timeout=60).raise_for_status()
tasks = requests.get(f"{HF_SPACE_BASE_URL}/tasks", timeout=60).json().get("tasks", [])
task_ids = [t.get("id") for t in tasks if t.get("id")]

print("Tasks:", task_ids)


Tasks: ['easy_standard_long_horizon', 'easy_heterogeneous_long_horizon', 'easy_fixed_obstacles_long_horizon', 'medium_standard_long_horizon', 'medium_heterogeneous_long_horizon', 'medium_fixed_obstacles_long_horizon', 'hard_standard_long_horizon', 'hard_heterogeneous_long_horizon', 'hard_fixed_obstacles_long_horizon']


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_load_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if torch.cuda.is_available():
    model_load_kwargs["device_map"] = "auto"
    model_load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

try:
    model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME, **model_load_kwargs)
    model_ref = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME, **model_load_kwargs)
except Exception as e:
    print("Primary 4-bit load failed, retrying without quantization:", e)
    fallback_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    }
    if torch.cuda.is_available():
        fallback_kwargs["device_map"] = "auto"
    model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME, **fallback_kwargs)
    model_ref = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME, **fallback_kwargs)

ppo_config = PPOConfig(
    batch_size=4,
    mini_batch_size=1,
    learning_rate=8e-6,
    log_with=None,
)

ppo_trainer = PPOTrainer(ppo_config, model, model_ref, tokenizer)
print("Model loaded:", MODEL_NAME)


In [13]:
def summarize_history(history, limit=4):
    return history[-limit:]


def compact_block_summary(summary):
    if not isinstance(summary, dict):
        return None
    return {
        "id": summary.get("id"),
        "priority": round(float(summary.get("priority_score", 0.0)), 3),
        "committed": bool(summary.get("committed", False)),
        "move_count": int(summary.get("move_count", 0) or 0),
        "placed_neighbors": [
            {
                "id": n.get("id"),
                "w": round(float(n.get("weight", 0.0)), 3),
                "pos": n.get("position"),
            }
            for n in (summary.get("placed_neighbors") or [])[:3]
            if isinstance(n, dict)
        ],
    }


def compact_density(density_map):
    if not isinstance(density_map, list) or not density_map:
        return {"mean": 0.0, "max": 0.0, "nonzero": 0}
    flat = []
    for row in density_map:
        if isinstance(row, list):
            for value in row:
                try:
                    flat.append(float(value))
                except (TypeError, ValueError):
                    pass
    if not flat:
        return {"mean": 0.0, "max": 0.0, "nonzero": 0}
    nonzero = sum(1 for v in flat if v > 0.0)
    return {
        "mean": round(sum(flat) / len(flat), 4),
        "max": round(max(flat), 4),
        "nonzero": int(nonzero),
    }


def build_prompt(obs, step, total_reward, recent_history, previous_failure=""):
    placed_blocks = list(getattr(obs, "placed_blocks", []) or [])
    remaining_blocks = list(getattr(obs, "remaining_blocks", []) or [])
    phase = str(getattr(obs, "phase", "placement") or "placement")
    instruction = str(getattr(obs, "instruction", "") or "")
    block_summaries = list(getattr(obs, "block_summaries", []) or [])[:6]
    placement_focus = getattr(obs, "placement_focus", None)
    candidate_actions = list(getattr(obs, "candidate_positions", []) or [])[:8]
    density_stats = compact_density(getattr(obs, "density_map", []))

    summaries_compact = [
        compact_block_summary(s) for s in block_summaries if isinstance(s, dict)
    ]
    focus_compact = compact_block_summary(placement_focus) if isinstance(placement_focus, dict) else None

    base = (
        "Choose exactly one action from candidates. Return JSON only.\n"
        "Valid formats: "
        "{\"block_id\":\"...\",\"x\":0,\"y\":0,\"action_type\":\"place\"} "
        "or action_type move/commit when appropriate.\n"
        "Use move only in repair/finalize; use commit when placement should be locked.\n"
        "Prefer legal actions that reduce wirelength and improve completion.\n\n"
        f"step={step}\n"
        f"phase={phase}\n"
        f"instruction={instruction}\n"
        f"placed_count={len(placed_blocks)} remaining_count={len(remaining_blocks)} total_reward={total_reward:.3f}\n"
        f"focus={json.dumps(focus_compact, ensure_ascii=False)}\n"
        f"summaries={json.dumps(summaries_compact, ensure_ascii=False)}\n"
        f"density_stats={json.dumps(density_stats, ensure_ascii=False)}\n"
        f"recent={json.dumps(summarize_history(recent_history, limit=4), ensure_ascii=False)}\n"
        f"candidates={json.dumps(candidate_actions, ensure_ascii=False)}\n"
    )
    if previous_failure:
        base += f"failure={previous_failure}\n"
    return base


In [14]:
def extract_json_object(text):
    text = (text or "").strip()
    if not text:
        return None
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        return None

    snippet = match.group(0)
    try:
        parsed = json.loads(snippet)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        try:
            parsed = ast.literal_eval(snippet)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return None
    return None


def normalize_action(action_data, obs, env):
    if not isinstance(action_data, dict):
        return None, "invalid_or_empty_model_output"

    block_id = action_data.get("block_id")
    x = action_data.get("x")
    y = action_data.get("y")
    action_type = str(action_data.get("action_type", "place") or "place").strip().lower()

    if not isinstance(block_id, str):
        return None, "invalid_block_id"
    if not isinstance(x, int) or not isinstance(y, int):
        return None, "invalid_coordinates"
    if action_type not in {"place", "move", "commit"}:
        return None, "invalid_action_type"

    candidates = list(getattr(obs, "candidate_positions", []) or [])
    allowed = {
        (
            str(c.get("block_id")),
            int(c.get("x")),
            int(c.get("y")),
            str(c.get("action_type", "place") or "place").lower(),
        )
        for c in candidates
        if isinstance(c, dict)
        and c.get("block_id") is not None
        and c.get("x") is not None
        and c.get("y") is not None
    }

    proposal = (block_id, x, y, action_type)
    if allowed and proposal not in allowed:
        return None, "action_not_in_candidates"

    if block_id not in env.block_id_to_index:
        return None, "unknown_block_id"

    action = {
        "x": int(x),
        "y": int(y),
        "choosen_block_index": int(env.block_id_to_index[block_id]),
        "action_type": action_type,
    }
    return action, ""


def parse_action(text, obs, env):
    parsed = extract_json_object(text)
    return normalize_action(parsed, obs, env)


In [ ]:
episode_rewards = []
episode_lengths = []
all_step_rewards = []

NUM_EPISODES = 8
MAX_STEPS = 30
MAX_MODEL_RETRIES = 3
PPO_BATCH_TRIGGER = 4
INVALID_OUTPUT_PENALTY = -0.20
REQUEST_ERROR_PENALTY = -0.30

for episode in range(NUM_EPISODES):
    task = random.choice(task_ids)
    obs = env.reset(task_name=task)

    total_reward = 0.0
    queries, responses, rewards = [], [], []
    recent_history = []
    previous_failure = ""

    print(f"\n=== Episode {episode + 1}/{NUM_EPISODES} | Task: {task} ===")

    for step in range(1, MAX_STEPS + 1):
        prompt = build_prompt(obs, step, total_reward, recent_history, previous_failure)
        query_tensor = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids[0].to(device)

        action = None
        parse_reason = ""

        for attempt in range(1, MAX_MODEL_RETRIES + 1):
            try:
                response_tensors = ppo_trainer.generate(
                    [query_tensor],
                    max_new_tokens=96,
                    do_sample=True,
                    temperature=0.2,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )
                response_tensor = response_tensors[0]
                response_text = tokenizer.decode(response_tensor, skip_special_tokens=True).strip()
                action, parse_reason = parse_action(response_text, obs, env)
            except Exception as e:
                response_tensor = None
                response_text = ""
                action = None
                parse_reason = f"generation_error:{type(e).__name__}"

            if action is not None:
                break

            previous_failure = parse_reason
            if attempt < MAX_MODEL_RETRIES:
                time.sleep(0.2)

        if action is None:
            total_reward += INVALID_OUTPUT_PENALTY
            all_step_rewards.append(INVALID_OUTPUT_PENALTY)
            recent_history.append({
                "step": step,
                "reward": round(INVALID_OUTPUT_PENALTY, 3),
                "error": parse_reason,
                "phase": str(getattr(obs, "phase", "placement") or "placement"),
            })
            recent_history = recent_history[-24:]
            print(f"Step {step}: invalid model output ({parse_reason})")
            continue

        try:
            next_obs = env.step(action)
        except Exception as e:
            step_reward = REQUEST_ERROR_PENALTY
            total_reward += step_reward
            all_step_rewards.append(step_reward)
            previous_failure = f"env_step_error:{type(e).__name__}"
            recent_history.append({
                "step": step,
                "reward": round(step_reward, 3),
                "error": previous_failure,
                "phase": str(getattr(obs, "phase", "placement") or "placement"),
            })
            recent_history = recent_history[-24:]
            print(f"Step {step}: env.step failed ({type(e).__name__})")
            continue

        reward = float(getattr(next_obs, "reward", 0.0) or 0.0)
        done = bool(getattr(next_obs, "done", False))
        invalid_reason = getattr(next_obs, "invalid_reasons", None)

        if invalid_reason:
            reward += INVALID_OUTPUT_PENALTY
            previous_failure = str(invalid_reason)
        else:
            previous_failure = ""

        total_reward += reward
        all_step_rewards.append(reward)

        queries.append(query_tensor)
        responses.append(response_tensor)
        rewards.append(torch.tensor(reward, dtype=torch.float32, device=device))

        if len(queries) >= PPO_BATCH_TRIGGER:
            try:
                ppo_trainer.step(queries, responses, rewards)
            except Exception as e:
                print("PPO step skipped due to error:", type(e).__name__)
            queries, responses, rewards = [], [], []

        recent_history.append({
            "step": step,
            "reward": round(reward, 3),
            "done": done,
            "phase": str(getattr(next_obs, "phase", "placement") or "placement"),
            "action": action,
        })
        recent_history = recent_history[-24:]

        print(f"Step {step}: reward={reward:.3f} done={done}")
        obs = next_obs

        if done:
            break

    if queries:
        try:
            ppo_trainer.step(queries, responses, rewards)
        except Exception as e:
            print("Final PPO flush skipped due to error:", type(e).__name__)

    episode_rewards.append(total_reward)
    episode_lengths.append(step)
    print(f"Episode {episode + 1} reward: {total_reward:.3f} | steps: {step}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(episode_rewards) + 1), episode_rewards, marker="o", linewidth=2)
axes[0].set_title("Episode Reward")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(episode_lengths) + 1), episode_lengths, marker="s", color="tab:orange", linewidth=2)
axes[1].set_title("Episode Length")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Steps")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
